# FIFA ANALYSIS

## CELL 1: Import libraries

In [1]:
import pandas as pd
import os
import sqlite3

## CELL 2: Set file paths 
### Update these paths to match where your CSVs are on your Mac

In [15]:
# Updating these paths to match where my CSVs are on my device
DATA_RAW = "../data/raw/"
 
paths = {
    "results"      : DATA_RAW + "results.csv",
    "wc_finals"    : DATA_RAW + "List of FIFA World Cup finals.csv",
    "wc_attendance": DATA_RAW + "FIFA World Cup Attendance.csv",
    "wc_awards"    : DATA_RAW + "FIFA World Cup Award.csv",
    "wc_top4"      : DATA_RAW + "Teams reaching the top four.csv",
    "fifa_ranking" : DATA_RAW + "fifa_ranking-2023-07-20.csv",}

for name, path in paths.items():
    status = "Loaded Successfully!" if os.path.exists(path) else "NOT loaded — check path"
    print(f"{name:20s} -> {status}")

results              -> Loaded Successfully!
wc_finals            -> Loaded Successfully!
wc_attendance        -> Loaded Successfully!
wc_awards            -> Loaded Successfully!
wc_top4              -> Loaded Successfully!
fifa_ranking         -> Loaded Successfully!


## CELL 3: Load all datasets 

In [3]:
results       = pd.read_csv(paths["results"],       encoding="latin-1")
wc_finals     = pd.read_csv(paths["wc_finals"],     encoding="latin-1")
wc_attendance = pd.read_csv(paths["wc_attendance"], encoding="latin-1")
wc_awards     = pd.read_csv(paths["wc_awards"],     encoding="latin-1")
wc_top4       = pd.read_csv(paths["wc_top4"],       encoding="latin-1")
fifa_ranking  = pd.read_csv(paths["fifa_ranking"],  encoding="latin-1")

print("Datasets loaded successfully!")

Datasets loaded successfully!


## CELL 4: Shape check — how many rows and columns? 

In [4]:
datasets = {
    "results"       : results,
    "wc_finals"     : wc_finals,
    "wc_attendance" : wc_attendance,
    "wc_awards"     : wc_awards,
    "wc_top4"       : wc_top4,
    "fifa_ranking"  : fifa_ranking,
}
# results.head()
# results.info()
# results.describe()
# results.isnull().sum()
# results['home_team'].value_counts().head(10)
# results['tournament'].unique()

print(f"{'Dataset':<20} {'Rows':>8} {'Columns':>10}")
for name, df in datasets.items():
    print(f"{name:<20} {df.shape[0]:>8,} {df.shape[1]:>10}")

Dataset                  Rows    Columns
results                43,281          9
wc_finals                  25         10
wc_attendance              23          9
wc_awards                  22          9
wc_top4                    25          7
fifa_ranking           64,757          8


## CELL 5: Column names of every dataset

In [5]:
for name, df in datasets.items():
    print(f"\n {name}")
    print(df.columns.tolist())


 results
['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'city', 'country', 'id']

 wc_finals
['Unnamed: 0', 'Year', 'Host', 'Champion', 'Score', 'Runner_up', 'Third', 'Score.1', 'Fourth', 'No. _ofteams']

 wc_attendance
['Unnamed: 0', 'Year', 'Hosts', 'Total_Attendance', 'Matches', 'Average_Attendance', 'Number', 'Venue', 'Game(s)']

 wc_awards
['Unnamed: 0', 'World _Cup', 'Golden _Ball', 'Golden _Boot', 'Goals', 'Golden _Glove', 'Clean _sheets', 'FIFA _Young _Player_ Award', 'FIFA _Fair _Play _Trophy']

 wc_top4
['Unnamed: 0', 'Team', 'Titles', 'Runners-up', 'Third place', 'Fourth place', 'Top 4 Total']

 fifa_ranking
['rank', 'country_full', 'country_abrv', 'total_points', 'previous_points', 'rank_change', 'confederation', 'rank_date']


## CELL 6: Null value check — is the data clean? 

In [6]:
print("MISSING VALUES PER COLUMN\n")
for name, df in datasets.items():
    nulls = df.isnull().sum()
    has_nulls = nulls[nulls > 0]
    if len(has_nulls) == 0:
        print(f"{name}: No nulls")
    else:
        print(f"{name}: Has nulls")
        print(has_nulls)
        print()

MISSING VALUES PER COLUMN

results: Has nulls
home_score    4
away_score    4
dtype: int64

wc_finals: Has nulls
Champion     1
Score        1
Runner_up    1
Third        1
Score.1      1
Fourth       1
dtype: int64

wc_attendance: Has nulls
Game(s)    1
dtype: int64

wc_awards: No nulls
wc_top4: Has nulls
Titles          17
Runners-up      15
Third place     11
Fourth place     8
dtype: int64

fifa_ranking: No nulls


## CELL 7: Data type check
### Dates should be datetime, not string — we'll fix this

In [7]:
# ── CELL 7: Data type check
# Dates should be datetime, not string — we'll fix this
for name, df in datasets.items():
    print(f"\n── {name} dtypes ──")
    print(df.dtypes)


── results dtypes ──
date              str
home_team         str
away_team         str
home_score    float64
away_score    float64
tournament        str
city              str
country           str
id              int64
dtype: object

── wc_finals dtypes ──
Unnamed: 0      int64
Year            int64
Host              str
Champion          str
Score             str
Runner_up         str
Third             str
Score.1           str
Fourth            str
No. _ofteams      str
dtype: object

── wc_attendance dtypes ──
Unnamed: 0            int64
Year                    str
Hosts                   str
Total_Attendance      int64
Matches               int64
Average_Attendance    int64
Number                  str
Venue                   str
Game(s)                 str
dtype: object

── wc_awards dtypes ──
Unnamed: 0                    int64
World _Cup                      str
Golden _Ball                    str
Golden _Boot                    str
Goals                         int64
Golden _Gl

## CELL 8: Convert date columns to datetime

In [8]:
results["date"] = pd.to_datetime(results["date"])
fifa_ranking["rank_date"] = pd.to_datetime(fifa_ranking["rank_date"])
 
print("Date columns converted")
print(f"results date range     : {results['date'].min().date()}     {results['date'].max().date()}")
print(f"fifa_ranking date range: {fifa_ranking['rank_date'].min().date()}     {fifa_ranking['rank_date'].max().date()}")

Date columns converted
results date range     : 1872-11-30     2023-11-28
fifa_ranking date range: 1992-12-31     2023-07-20


## CELL 9: Quick look at each dataset (first 3 rows)

In [9]:
for name, df in datasets.items():
    print(f"\n{'-'*180}")
    print(f"  {name}")
    print(f"{'-'*180}")
    print(df.head(3).to_string())


------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
  results
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
        date home_team away_team  home_score  away_score tournament     city   country  id
0 1872-11-30  Scotland   England         0.0         0.0   Friendly  Glasgow  Scotland   1
1 1873-03-08   England  Scotland         4.0         2.0   Friendly   London   England   2
2 1874-03-07  Scotland   England         2.0         1.0   Friendly  Glasgow  Scotland   3

------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
  wc_finals
---------------------------------------------------------------------

## CELL 10: results.csv — tournament breakdown
### This shows you every tournament type in the dataset


In [10]:
print("All TOURNAMENT types in results.csv:\n")
print(results["tournament"].value_counts().to_string())

All TOURNAMENT types in results.csv:

tournament
Friendly                                         17151
FIFA World Cup qualification                      8012
UEFA Euro qualification                           2815
African Cup of Nations qualification              1998
FIFA World Cup                                     964
Copa AmÃ©rica                                      841
AFC Asian Cup qualification                        764
African Cup of Nations                             741
Merdeka Tournament                                 572
British Home Championship                          517
CECAFA Cup                                         502
UEFA Nations League                                472
CFU Caribbean Cup qualification                    398
Gulf Cup                                           395
AFC Asian Cup                                      370
Gold Cup                                           345
UEFA Euro                                          337
COSAFA Cup      

## CELL 11: results.csv — filter for key tournaments

In [11]:
key_tournaments = [
    "FIFA World Cup",
    "Copa América",
    "UEFA Euro",
    "FIFA World Cup qualification",
    "UEFA Euro qualification",
    "CONMEBOL Copa América",         # older name used in some records
]

key_matches = results[results["tournament"].isin(key_tournaments)]
print(f"Key tournament matches: {len(key_matches):,} out of {len(results):,} total")
print()
print(key_matches["tournament"].value_counts())

Key tournament matches: 12,128 out of 43,281 total

tournament
FIFA World Cup qualification    8012
UEFA Euro qualification         2815
FIFA World Cup                   964
UEFA Euro                        337
Name: count, dtype: int64


## CELL 12: fifa_ranking.csv — unique teams and confederations

In [12]:
print(f"Unique teams in ranking data: {fifa_ranking['country_full'].nunique()}")
print()
print("Teams per confederation:")
print(fifa_ranking.drop_duplicates("country_full")["confederation"].value_counts())

Unique teams in ranking data: 231

Teams per confederation:
confederation
UEFA        61
CAF         60
AFC         47
CONCACAF    41
OFC         12
CONMEBOL    10
Name: count, dtype: int64


## CELL 13: fifa_ranking — pre-WC snapshots check
### For the ML model we need rankings just before each WC
### Check what dates we have close to each WC June

In [13]:
# For the ML model we need rankings just before each WC
# Check what dates we have close to each WC June
wc_years = [1994, 1998, 2002, 2006, 2010, 2014, 2018, 2022]
 
print("Pre-tournament ranking snapshots available:\n")
print(f"{'WC Year':<10} {'Closest ranking date':<25} {'Teams ranked'}")
print("-" * 50)
 
for yr in wc_years:
    may_june = fifa_ranking[(fifa_ranking["rank_date"].dt.year == yr) &
        (fifa_ranking["rank_date"].dt.month.isin([4, 5, 6]))]
    
    if len(may_june) > 0:
        latest = may_june["rank_date"].max()
        n_teams = fifa_ranking[fifa_ranking["rank_date"] == latest].shape[0]
        print(f"{yr:<10} {str(latest.date()):<25} {n_teams}")
    else:
        print(f"{yr:<10} {'No April–June date found':<25} —")

Pre-tournament ranking snapshots available:

WC Year    Closest ranking date      Teams ranked
--------------------------------------------------
1994       1994-06-14                159
1998       1998-05-20                187
2002       2002-05-15                202
2006       2006-05-17                204
2010       2010-05-26                201
2014       2014-06-05                206
2018       2018-06-07                205
2022       2022-06-23                211


## CELL 14: wc_finals — champions list 

In [14]:
print("World Cup Champions (1930–2022):\n")
# Drop the messy unnamed index column first
wc_clean = wc_finals.drop(columns=["Unnamed: 0"], errors="ignore")
print(wc_clean[["Year", "Host", "Champion", "Runner_up"]].to_string(index=False))

World Cup Champions (1930–2022):

 Year                               Host                           Champion                          Runner_up
 1930                            Uruguay                            Uruguay                          Argentina
 1934                              Italy                              Italy                     Czechoslovakia
 1938                             France                              Italy                            Hungary
 1942 (Not held because of World War II) (Not held because of World War II) (Not held because of World War II)
 1946 (Not held because of World War II) (Not held because of World War II) (Not held because of World War II)
 1950                             Brazil                            Uruguay                             Brazil
 1954                        Switzerland                       West Germany                            Hungary
 1958                             Sweden                             Brazil   


#### results.csv:
  1. Drop columns: city, id
  2. Ends Nov 2023 — need 2024-2025 update
  3. Copa América encoded as 'Copa AmÃ©rica' — fix encoding
  4. Create: result column (H/A/D) from scores
 
#### wc_finals.csv:
  1. Drop: Unnamed: 0, Score, Score.1 (venue mixed in)
  2. Rename: 'No. _ofteams' -> 'num_teams'
 
#### wc_top4.csv:
  1. 'Germany1' -> 'Germany'
  2. Extract numbers from 'Titles' column (e.g. "5 (1958...)" ->5)
 
#### fifa_ranking.csv:
  1. Drop: previous_points, rank_change
  2. Fix team name mismatches with results.csv (e.g. 'Korea Republic' vs 'South Korea')
  3. Ends July 2023 — need 2024-2026 update for ML
 
# These are exactly what Notebook 02 (SQL cleaning) will fix.